# Create VAE for the different words sin disntace functions

In [ ]:
# Add import 
import sys
import torch 
from torch import nn
from torch import optim
from prodigyopt import Prodigy # proddigy optimizer from https://github.com/konstmish/prodigy?tab=readme-ov-file
import lightning.pytorch as pl

import tqdm
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
# allow reload of python modules
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
from dataset.RobotPathDataset.normalizer import MinMaxFeatureNormalizer

from model.models import EMA
import copy
import time

# remove all warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
normalizer = MinMaxFeatureNormalizer()
encoder_normalizer = MinMaxFeatureNormalizer()
representation_dim = 64
CONFIG = {
    # Model configuration
    'representation_dim': representation_dim,
    'dim_mults':(1,4,8), # Dimension multipliers for hidden layers of the model # (1, 4, 8)
    
    # Encoder configuration
    'embeder_num_of_hidden_layers' : 1,
    'vae_kwargs': {
        'in_channels':1,
        'latent_dim': representation_dim,
        'hidden_dims': [4, 8, 16, 32, 64],
    },
    
    # Hyperparameters for training
    'batch_size': 8,
    'num_epochs': 20,
    'ema': EMA(beta=0.99), # Exponential moving average for the model weights
        ## Optimizer configuration
        'optimizer': Prodigy,
        'optimizer_kwargs': {
            'lr': 1., # ! ONLY FOR PRODIGY OPTIMIZER
            'weight_decay': 0.01, 
            'safeguard_warmup':True,
            'use_bias_correction':True,
            'betas': (0.9, 0.99),
            },
        # Scheduler configuration
        'scheduler': torch.optim.lr_scheduler.CosineAnnealingLR,
        'scheduler_kwargs': {
            # 'gamma': 0.999,
            'T_max': 10, # Total number of iterations
        },
    
    # Hyperparameters for diffusion process
    'noise_steps': 256,
    'normalize': True,
    'normalizer':normalizer,
    'encoder_normalizer':encoder_normalizer,
    'cfg_scale': 3,


    # Dataset specific configuration
    'n_paths_per_world': 10,
    'n_worlds': 10,
    'n_waypoints': 64, # due to the archtechture has to be a number that is a power of 2 
}

# DataLoader

### Use the other dataset class to ready the data then get the worlds images from it

In [ ]:
from dataset import RobotPathDataset
file = 'data/SingleSphere02_all.db'
# file = 'D:\Desktop\ADLR\RobotPathData\data\SingleSphere02_one-world.db'
dataset = RobotPathDataset(file, n_paths_per_world=CONFIG['n_paths_per_world'], n_worlds=CONFIG['n_worlds'],  n_waypoints=CONFIG['n_waypoints'],   normalizer=CONFIG['normalizer'])
print(f' sample shape {dataset[0]["path"].shape} with {len(dataset)} samples')

In [ ]:
from dataset.RobotPathDataset.obstacle_distance import img2dist_img

In [ ]:
class WorldsDataset(torch.utils.data.Dataset):
    def __init__(self, dataset:RobotPathDataset, sign_dist_field=False):
        worlds_dataset = dataset.all_world_images
        self.data = []
        
        self.voxel_size = 10 / 64     # in m

        if sign_dist_field:
            for i in range(len(worlds_dataset)):
                dist_field= torch.tensor(img2dist_img(img=worlds_dataset[i], voxel_size=self.voxel_size, add_boundary=True), dtype=torch.float32)
                self.data.append(dist_field)
                
        self.data = torch.stack(self.data)
        self.data = self.data.unsqueeze(1)
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

In [ ]:
world_dataset = WorldsDataset(dataset, sign_dist_field=True)
print(f' sample shape {world_dataset[0].shape} with {len(world_dataset)} samples')

In [ ]:
sample = world_dataset[9]
sample

In [ ]:
plt.imshow(sample.T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')

### Dataloader

In [ ]:
from torch.utils.data.dataset import random_split
train_dataset, val_dataset = random_split(world_dataset, [0.9, 0.1])

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=True)
sample_batch = next(iter(train_dataloader))

# Model

In [ ]:
from model.vae import VanillaVAE, BaseVAE

In [ ]:
normalizer = MinMaxFeatureNormalizer()
model = VanillaVAE(**CONFIG['vae_kwargs'], normalizer=normalizer)
model

In [ ]:
normalized_sample_batch = normalizer.normalize(sample_batch)

In [ ]:
mu, log_var = model.encode(sample_batch)
z = model.reparameterize(mu, log_var)
reconstructed = model.decode(z)
assert reconstructed.shape == sample_batch.shape , f'{reconstructed.shape} != {sample_batch.shape}'

In [ ]:
results = model.forward(sample_batch)
results
model.loss_function(*results, M_N=1)

In [ ]:
# visualize const vs reconstructed
fig, ax = plt.subplots(1,3, figsize=(10,5))
pos = ax[0].imshow(sample_batch[0].squeeze().T, origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[0].set_title('Original')
fig.colorbar(pos, ax=ax[0])
pos = ax[1].imshow(reconstructed[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[1].set_title('Reconstructed')
fig.colorbar(pos, ax=ax[1])
pos = ax[2].imshow(normalized_sample_batch[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[2].set_title('Normalized')
fig.colorbar(pos, ax=ax[2])

In [ ]:
from model.vae import VanillaVAE, BaseVAE, VAEXperiment
autoencoder = VAEXperiment(model, CONFIG['vae_kwargs'])

In [ ]:
# DEBUG lightning before running full training
trainer = pl.Trainer(limit_train_batches=64, max_epochs=2)
trainer.fit(model=autoencoder, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

In [ ]:
import os
from lightning.pytorch import loggers
logger = loggers.TensorBoardLogger(os.getcwd(), name='vae')
trainer = pl.Trainer(enable_progress_bar=True, enable_checkpointing=True, logger=logger, max_epochs=CONFIG['num_epochs'])
trainer.fit(model=autoencoder, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

In [ ]:
model = autoencoder.model
model.eval()
sample = world_dataset[9]
sample = sample.unsqueeze(0)
# sample = sample.to(autoencoder.curr_device)
mu, log_var = model.encode(sample)
z = model.reparameterize(mu, log_var)
reconstructed = model.decode(z)
assert reconstructed.shape == sample.shape , f'{reconstructed.shape} != {sample.shape}'
# visualize const vs reconstructed
fig, ax = plt.subplots(1,2, figsize=(10,5))
pos = ax[0].imshow(sample[0].squeeze().T.cpu().detach(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[0].set_title('Original')
fig.colorbar(pos, ax=ax[0])
pos = ax[1].imshow(reconstructed[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[1].set_title('Reconstructed')
fig.colorbar(pos, ax=ax[1])


In [ ]:
# reload model form checkpoint
model_old = VAEXperiment.load_from_checkpoint(r'/home/karim.samir.lotfy/tum-adlr-ss24-09/lightning_logs/version_10/checkpoints/epoch=227-step=256500.ckpt')

In [ ]:
model_old.freeze()
vae_model = model_old.model


In [ ]:
vae_model.eval()
sample = world_dataset[88]
sample = sample.unsqueeze(0)
sample = sample.to(autoencoder.curr_device)
mu, log_var = model.encode(sample)
z = model.reparameterize(mu, log_var)
reconstructed = model.decode(z)
assert reconstructed.shape == sample.shape , f'{reconstructed.shape} != {sample.shape}'
# visualize const vs reconstructed
fig, ax = plt.subplots(1,2, figsize=(10,5))
pos = ax[0].imshow(sample[0].squeeze().T.cpu().detach(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[0].set_title('Original')
fig.colorbar(pos, ax=ax[0])
pos = ax[1].imshow(reconstructed[0].squeeze().T.cpu().detach().numpy(), origin='lower', extent=[0, 10, 0, 10], cmap='viridis')
ax[1].set_title('Reconstructed')
fig.colorbar(pos, ax=ax[1])

In [ ]:
plt.show()